#### BERT

In [1]:
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
import torch

C:\Users\PC\anaconda3\envs\llms_course_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
bert_model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
bert_model = BertForQuestionAnswering.from_pretrained(bert_model_name)

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
# Data
sunset_motors_context = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retail. Sunset Motors proudly offers a diverse range of vehicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMW, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexible financing options, allowing customers to secure affordable loans and leases with competitive interest rates."
print(sunset_motors_context)

Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the picturesque town of Crestwood, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for excellence, reliability, and customer satisfaction over the past four decades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned business with a small lot of used cars. However, under Anderson's leadership and commitment to quality, it quickly evolved into a thriving dealership offering a wide range of vehicles from various manufacturers. Today, the dealership spans over 10 acres, showcasing a vast inventory of new and pre-owned cars, trucks, SUVs, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustainability. In 2010, the dealership made a landmark decision to incorporate environmentally friendly practices, including solar p

In [4]:
# Feed question
def faq_bot(question):

    context = sunset_motors_context
    input_ids = bert_tokenizer.encode(question, context)
    tokens = bert_tokenizer.convert_ids_to_tokens(input_ids)
    
    # Create segment embeddings
    sep_idx = input_ids.index(bert_tokenizer.sep_token_id)
    num_seg_a = sep_idx+1
    num_seg_b = len(input_ids) - num_seg_a
    segment_ids = [0]*num_seg_a + [1]*num_seg_b
    
    # Feed the input into the model
    output = bert_model(torch.tensor([input_ids]), token_type_ids = torch.tensor([segment_ids]))

    # Get predicted answer span
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)

    # Extract raw answer span
    if answer_end >= answer_start:
        answer = ' '.join(tokens[answer_start:answer_end+1])
    else:
        print("I don't know how to answer this question, can you ask another one?")

    # Fix subword tokens
    corrected_answer = ''
    for word in answer.split():
        if word[0:2] == '##':
            corrected_answer += word[2:]
        else:
            corrected_answer += ' ' + word
    return corrected_answer

In [5]:
faq_bot("Where is the dealership located?")

' crestwood'

In [6]:
faq_bot("What make of cars are available?")

' ford , toyota , honda , chevrolet , and bmw'

In [7]:
faq_bot("how large is the dealership?") 

' 10 acres'

#### RoBERTa

In [8]:
from transformers import RobertaTokenizer, RobertaForQuestionAnswering
robert_model_name = "deepset/roberta-base-squad2"
roberta_tokenizer = RobertaTokenizer.from_pretrained(robert_model_name)
roberta_model = RobertaForQuestionAnswering.from_pretrained(robert_model_name)

In [9]:
# Feed question
def roberta_faq_bot(question):

    context = sunset_motors_context
    input_ids = roberta_tokenizer.encode(question, context)
    tokens = roberta_tokenizer.convert_ids_to_tokens(input_ids)
    
    # Feed the input into the model
    output = roberta_model(torch.tensor([input_ids]))

    # Get predicted answer span
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)

    # Extract raw answer span
    if answer_end >= answer_start:
        answer_tokens = tokens[answer_start:answer_end+1]
        return roberta_tokenizer.convert_tokens_to_string(answer_tokens)
    else:
        return "I don't know how to answer this question, can you ask another one?"

In [10]:
roberta_faq_bot("Where is the dealership located?")

' Crestwood'

In [11]:
roberta_faq_bot("What make of cars are available?")

' new and pre-owned cars, trucks, SUVs, and luxury vehicles'

In [12]:
roberta_faq_bot("How large is the dealership?") 

' over 10 acres'

#### DistilBERT

In [13]:
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering
distilbert_model_name = "distilbert-base-uncased-distilled-squad"
distilbert_tokenizer = DistilBertTokenizer.from_pretrained(distilbert_model_name)
distilbert_model = DistilBertForQuestionAnswering.from_pretrained(distilbert_model_name)

In [14]:
# Feed question
def distilbert_faq_bot(question):

    context = sunset_motors_context
    input_ids = distilbert_tokenizer.encode(question, context)
    tokens = distilbert_tokenizer.convert_ids_to_tokens(input_ids)
    
    # Feed the input into the model
    output = distilbert_model(torch.tensor([input_ids]))

    # Get predicted answer span
    answer_start = torch.argmax(output.start_logits)
    answer_end = torch.argmax(output.end_logits)

    # Extract raw answer span
    if answer_end >= answer_start:
        answer_tokens = tokens[answer_start:answer_end+1]
        return distilbert_tokenizer.convert_tokens_to_string(answer_tokens)
    else:
        return "I don't know how to answer this question, can you ask another one?"

In [15]:
distilbert_faq_bot("Where is the dealership located?")

'crestwood'

In [16]:
distilbert_faq_bot("What make of cars are available?")

'used cars'

In [17]:
distilbert_faq_bot("How large is the dealership?") 

'10 acres'